In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
# LOAD DATA
train_path = "mnist_train.csv"
train_df = pd.read_csv(train_path)

if "label" in train_df.columns:
    X = train_df.drop("label", axis=1).values
    y = train_df["label"].values
else:
    X = train_df.values[:, 1:]
    y = train_df.values[:, 0]

print("Train shape:", X.shape)

In [ ]:
# NORMALIZATION
X = X / 255.0

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# SCALING
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# PCA (Dimensionality Reduction for Classical ML)
pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("Reduced dimension:", X_train_pca.shape[1])

# 🧪 Task 1: Classical ML Models (Scikit-Learn)

We use KNN, SVM, and Decision Trees to form a high-accuracy ensemble.

In [ ]:
# -------- KNN --------
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_pca, y_train)
knn_pred = knn.predict(X_test_pca)

# -------- SVM (RBF Kernel) --------
svm = SVC(kernel="rbf", C=5, gamma="scale")
svm.fit(X_train_pca, y_train)
svm_pred = svm.predict(X_test_pca)

# -------- Decision Tree --------
dt = DecisionTreeClassifier(max_depth=15, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [ ]:
# EVALUATION
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, cmap="Blues", annot=True, fmt='d')
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

evaluate("KNN", y_test, knn_pred)
evaluate("SVM", y_test, svm_pred)
evaluate("Decision Tree", y_test, dt_pred)

# 🛠️ Task 2: Model Implementation From Scratch

As per assignment requirements, we implement **K-Nearest Neighbors (KNN)** from scratch using only NumPy. This demonstrates the underlying logic of distance-based classification.

In [ ]:
class KNNFromScratch:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X_test):
        predictions = [self._predict_one(x) for x in X_test]
        return np.array(predictions)

    def _predict_one(self, x):
        # 1. Compute Euclidean distance
        distances = np.linalg.norm(self.X_train - x, axis=1)
        
        # 2. Get indices of k nearest neighbors
        k_indices = np.argsort(distances)[:self.k]
        
        # 3. Get labels of k nearest neighbors
        k_nearest_labels = [self.y_train[i] for i in k_indices]
        
        # 4. Majority Vote
        most_common = np.bincount(k_nearest_labels).argmax()
        return most_common

# --- Demo on small subset (100 samples) for verification ---
X_small = X_test_pca[:100]
y_small = y_test[:100]

scratch_knn = KNNFromScratch(k=3)
scratch_knn.fit(X_train_pca, y_train)
scratch_pred = scratch_knn.predict(X_small)

accuracy = np.mean(scratch_pred == y_small)
print(f"KNN From Scratch Accuracy (on sample): {accuracy * 100:.2f}%")